In [ ]:
import pandas as pd

# Load existing ATM dataset
df = pd.read_csv("atms.csv")

# Display first 5 records
print("First 5 ATM records:")
print(df.head())

# Display dataset size
print("\nDataset shape:")
print(df.shape)

# Display column names
print("\nATM columns:")
print(df.columns.tolist())

First 5 ATM records:
   id     atm_id            bank_name       city   latitude  longitude  \
0   1  ATM100001            Axis Bank    Kolkata  22.595947  88.375225   
1   2  ATM100002  Kotak Mahindra Bank  Bangalore  13.049442  77.611524   
2   3  ATM100003  Bank of Maharashtra  Hyderabad  17.321285  78.431157   
3   4  ATM100004            Axis Bank       Pune  18.466109  73.806551   
4   5  ATM100005            HDFC Bank     Nagpur  21.093540  79.151682   

   highway_distance  lighting_score  cctv_coverage  historical_fraud_count  \
0              3.63              46             69                      47   
1              2.45              73             58                      26   
2              2.53              98             68                      20   
3              2.92              47             49                      21   
4              0.89              80             38                      34   

   withdrawal_limit  risk_score risk_level  
0             20000 

In [ ]:
print(df.columns.tolist())

['id', 'atm_id', 'bank_name', 'city', 'latitude', 'longitude', 'highway_distance', 'lighting_score', 'cctv_coverage', 'historical_fraud_count', 'withdrawal_limit', 'risk_score', 'risk_level']


In [ ]:
# --------------------------------
# STEP 2: Check ATM Data
# --------------------------------

# Check missing values
print("\nMissing values:")
print(df.isnull().sum())


# Check duplicate rows
print("\nDuplicate rows:")
print(df.duplicated().sum())


# Check data types
print("\nData types:")
print(df.dtypes)


# Check unique risk levels
print("\nRisk levels:")
print(df["risk_level"].value_counts())


# Basic statistics
print("\nBasic statistics:")
print(df.describe())


Missing values:
id                        0
atm_id                    0
bank_name                 0
city                      0
latitude                  0
longitude                 0
highway_distance          0
lighting_score            0
cctv_coverage             0
historical_fraud_count    0
withdrawal_limit          0
risk_score                0
risk_level                0
dtype: int64

Duplicate rows:
0

Data types:
id                          int64
atm_id                     object
bank_name                  object
city                       object
latitude                  float64
longitude                 float64
highway_distance          float64
lighting_score              int64
cctv_coverage               int64
historical_fraud_count      int64
withdrawal_limit            int64
risk_score                float64
risk_level                 object
dtype: object

Risk levels:
risk_level
MEDIUM    250
HIGH      134
LOW       116
Name: count, dtype: int64

Basic statistics:
      

In [ ]:
features = [
    "highway_distance",
    "lighting_score",
    "cctv_coverage",
    "historical_fraud_count",
    "withdrawal_limit"
]

In [ ]:
target = "risk_level"

In [ ]:
# --------------------------------
# STEP 3: Select ML Features
# --------------------------------

features = [
    "highway_distance",
    "lighting_score",
    "cctv_coverage",
    "historical_fraud_count",
    "withdrawal_limit"
]

target = "risk_level"


# Input data
X = df[features]

# Target/output
y = df[target]


print("\nSelected features:")
print(features)

print("\nTarget:")
print(target)

print("\nX shape:")
print(X.shape)

print("\ny shape:")
print(y.shape)


Selected features:
['highway_distance', 'lighting_score', 'cctv_coverage', 'historical_fraud_count', 'withdrawal_limit']

Target:
risk_level

X shape:
(500, 5)

y shape:
(500,)


In [ ]:
# --------------------------------
# STEP 4: Encode Target
# --------------------------------

from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

y_encoded = encoder.fit_transform(y)


print("\nOriginal risk levels:")
print(y.unique())

print("\nEncoded risk levels:")
print(y_encoded[:10])

print("\nClass mapping:")
for number, label in enumerate(encoder.classes_):
    print(number, "=", label)


Original risk levels:
['HIGH' 'MEDIUM' 'LOW']

Encoded risk levels:
[0 2 1 2 2 0 2 1 2 1]

Class mapping:
0 = HIGH
1 = LOW
2 = MEDIUM


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

Training data: (400, 5)
Testing data: (100, 5)
Training target: (400,)
Testing target: (100,)


In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Create Random Forest model
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Train the model
model.fit(X_train, y_train)

print("Random Forest model trained successfully!")

Random Forest model trained successfully!


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

# Make predictions on test data
y_pred = model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)

print("Model Accuracy:", accuracy)

# Detailed classification report
print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=encoder.classes_
))

Model Accuracy: 0.89

Classification Report:
              precision    recall  f1-score   support

        HIGH       1.00      0.81      0.90        27
         LOW       0.95      0.78      0.86        23
      MEDIUM       0.83      0.98      0.90        50

    accuracy                           0.89       100
   macro avg       0.93      0.86      0.88       100
weighted avg       0.90      0.89      0.89       100



In [ ]:
# Get feature importance
importance = model.feature_importances_

# Display feature importance
print("Feature Importance:")

for feature, value in zip(features, importance):
    print(feature, ":", round(value, 4))

Feature Importance:
highway_distance : 0.0977
lighting_score : 0.1247
cctv_coverage : 0.1139
historical_fraud_count : 0.6339
withdrawal_limit : 0.0299


In [ ]:
# Predict risk level for all ATMs
all_predictions = model.predict(X)

# Convert numeric predictions back to risk labels
predicted_risk_level = encoder.inverse_transform(all_predictions)

# Add predictions to a copy of the ATM dataset
atm_predictions = df.copy()

atm_predictions["predicted_risk_level"] = predicted_risk_level

# Display first 10 predictions
print(atm_predictions[
    ["atm_id", "risk_level", "predicted_risk_level"]
].head(10))

      atm_id risk_level predicted_risk_level
0  ATM100001       HIGH                 HIGH
1  ATM100002     MEDIUM               MEDIUM
2  ATM100003        LOW               MEDIUM
3  ATM100004     MEDIUM               MEDIUM
4  ATM100005     MEDIUM               MEDIUM
5  ATM100006       HIGH                 HIGH
6  ATM100007     MEDIUM               MEDIUM
7  ATM100008        LOW                  LOW
8  ATM100009     MEDIUM               MEDIUM
9  ATM100010        LOW                  LOW


In [ ]:
# Get prediction probabilities
probabilities = model.predict_proba(X)

# Get highest probability for each ATM
prediction_confidence = probabilities.max(axis=1)

# Convert confidence to percentage
prediction_confidence = prediction_confidence * 100

# Add confidence to predictions
atm_predictions["prediction_confidence"] = prediction_confidence.round(2)

# Display results
print(atm_predictions[
    ["atm_id", "risk_level", "predicted_risk_level", "prediction_confidence"]
].head(10))

      atm_id risk_level predicted_risk_level  prediction_confidence
0  ATM100001       HIGH                 HIGH                   96.0
1  ATM100002     MEDIUM               MEDIUM                   96.0
2  ATM100003        LOW               MEDIUM                   67.0
3  ATM100004     MEDIUM               MEDIUM                   97.0
4  ATM100005     MEDIUM               MEDIUM                   96.0
5  ATM100006       HIGH                 HIGH                   87.0
6  ATM100007     MEDIUM               MEDIUM                   88.0
7  ATM100008        LOW                  LOW                   82.0
8  ATM100009     MEDIUM               MEDIUM                   93.0
9  ATM100010        LOW                  LOW                   96.0


In [ ]:
# Load existing complaints dataset
complaints = pd.read_csv("complaints.csv")

# Show columns
print("Complaint columns:")
print(complaints.columns.tolist())

# Show first 5 records
print("\nFirst 5 complaint records:")
print(complaints.head())

Complaint columns:
['id', 'complaint_number', 'complainant_name', 'phone_number', 'bank_name', 'account_number', 'transaction_id', 'fraud_amount', 'transaction_type', 'fraud_date', 'fraud_time', 'location', 'description', 'source', 'created_at']

First 5 complaint records:
   id complaint_number complainant_name  phone_number            bank_name  \
0   1        CMP100001      Meera Patil    9583156434            Axis Bank   
1   2        CMP100002     Aditya Patil    9656307931            HDFC Bank   
2   3        CMP100003    Amit Kulkarni    9310777576  Kotak Mahindra Bank   
3   4        CMP100004      Sneha Gupta    9265564833            HDFC Bank   
4   5        CMP100005       Isha Gupta    9547306651  Bank of Maharashtra   

  account_number transaction_id  fraud_amount transaction_type  fraud_date  \
0       XXXX9476   TXN469919929      82165.64             IMPS  2026-08-21   
1       XXXX5982   TXN276799196      43543.42             IMPS  2026-04-29   
2       XXXX5496   TXN2

In [ ]:
# Convert complaint date and time
complaints["fraud_date"] = pd.to_datetime(complaints["fraud_date"])

complaints["fraud_time"] = pd.to_datetime(
    complaints["fraud_time"],
    format="%H:%M:%S"
).dt.time

# Add date
atm_predictions["date"] = complaints["fraud_date"].dt.strftime("%Y-%m-%d")

# Add time
atm_predictions["time"] = complaints["fraud_time"].astype(str)

# Create time of day
def get_time_of_day(time):
    hour = time.hour

    if 5 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 21:
        return "Evening"
    else:
        return "Night"

atm_predictions["time_of_day"] = complaints["fraud_time"].apply(
    get_time_of_day
)

# Check the result
print(
    atm_predictions[
        ["atm_id", "date", "time", "time_of_day"]
    ].head(10)
)

      atm_id        date      time time_of_day
0  ATM100001  2026-08-21  10:39:07     Morning
1  ATM100002  2026-04-29  22:03:07       Night
2  ATM100003  2026-08-06  18:25:07     Evening
3  ATM100004  2026-08-03  04:41:07       Night
4  ATM100005  2026-09-08  02:47:07       Night
5  ATM100006  2026-05-29  05:56:07     Morning
6  ATM100007  2026-05-18  19:09:07     Evening
7  ATM100008  2026-06-16  16:00:07   Afternoon
8  ATM100009  2026-05-29  14:23:07   Afternoon
9  ATM100010  2026-08-01  18:02:07     Evening


In [ ]:
cases = pd.read_csv("cases.csv")

print("Cases columns:")
print(cases.columns.tolist())

Cases columns:
['id', 'case_number', 'complaint_id', 'priority', 'status', 'assigned_to', 'risk_level', 'created_at', 'updated_at']


In [ ]:
atm_predictions["crime_type"] = complaints["description"]

print(
    atm_predictions[
        ["atm_id", "date", "time", "time_of_day", "crime_type"]
    ].head(10)
)

      atm_id        date      time time_of_day  \
0  ATM100001  2026-08-21  10:39:07     Morning   
1  ATM100002  2026-04-29  22:03:07       Night   
2  ATM100003  2026-08-06  18:25:07     Evening   
3  ATM100004  2026-08-03  04:41:07       Night   
4  ATM100005  2026-09-08  02:47:07       Night   
5  ATM100006  2026-05-29  05:56:07     Morning   
6  ATM100007  2026-05-18  19:09:07     Evening   
7  ATM100008  2026-06-16  16:00:07   Afternoon   
8  ATM100009  2026-05-29  14:23:07   Afternoon   
9  ATM100010  2026-08-01  18:02:07     Evening   

                        crime_type  
0            Online shopping fraud  
1             Phishing transaction  
2                  Fake KYC update  
3         Fake customer care fraud  
4                 Investment fraud  
5       Unauthorized bank transfer  
6                  Fake KYC update  
7  Remote access application fraud  
8                  Fake KYC update  
9                OTP sharing fraud  


In [ ]:
print("Final dataset shape:")
print(atm_predictions.shape)

print("\nMissing values:")
print(atm_predictions.isnull().sum())

print("\nFinal columns:")
print(atm_predictions.columns.tolist())

Final dataset shape:
(500, 19)

Missing values:
id                        0
atm_id                    0
bank_name                 0
city                      0
latitude                  0
longitude                 0
highway_distance          0
lighting_score            0
cctv_coverage             0
historical_fraud_count    0
withdrawal_limit          0
risk_score                0
risk_level                0
predicted_risk_level      0
prediction_confidence     0
date                      0
time                      0
time_of_day               0
crime_type                0
dtype: int64

Final columns:
['id', 'atm_id', 'bank_name', 'city', 'latitude', 'longitude', 'highway_distance', 'lighting_score', 'cctv_coverage', 'historical_fraud_count', 'withdrawal_limit', 'risk_score', 'risk_level', 'predicted_risk_level', 'prediction_confidence', 'date', 'time', 'time_of_day', 'crime_type']


In [ ]:
atm_predictions.to_csv("atm_predictions.csv", index=False)

print("atm_predictions.csv created successfully!")

atm_predictions.csv created successfully!


In [ ]:
import pandas as pd

# Load existing complaints
complaints = pd.read_csv("complaints.csv")

mule_hops = []

for idx, complaint in complaints.reset_index(drop=True).iterrows():

    complaint_id = complaint["complaint_number"]
    original_amount = complaint["fraud_amount"]
    fraud_timestamp = pd.to_datetime(complaint["created_at"])

    # Create accounts
    victim_account = f"ACC{idx+1:04d}V"
    mule_account_1 = f"ACC{idx+1:04d}M1"
    mule_account_2 = f"ACC{idx+1:04d}M2"
    cashout_account = f"ACC{idx+1:04d}M3"

    # Hop 1
    mule_hops.append({
        "hop_id": f"HOP{idx*3+1:04d}",
        "complaint_id": complaint_id,
        "hop_level": 1,
        "from_account": victim_account,
        "to_account": mule_account_1,
        "amount_transferred_inr": round(original_amount * 0.95, 2),
        "time_delay_minutes": 5,
        "timestamp": fraud_timestamp
    })

    # Hop 2
    mule_hops.append({
        "hop_id": f"HOP{idx*3+2:04d}",
        "complaint_id": complaint_id,
        "hop_level": 2,
        "from_account": mule_account_1,
        "to_account": mule_account_2,
        "amount_transferred_inr": round(original_amount * 0.90, 2),
        "time_delay_minutes": 12,
        "timestamp": fraud_timestamp + pd.Timedelta(minutes=17)
    })

    # Hop 3
    mule_hops.append({
        "hop_id": f"HOP{idx*3+3:04d}",
        "complaint_id": complaint_id,
        "hop_level": 3,
        "from_account": mule_account_2,
        "to_account": cashout_account,
        "amount_transferred_inr": round(original_amount * 0.85, 2),
        "time_delay_minutes": 8,
        "timestamp": fraud_timestamp + pd.Timedelta(minutes=25)
    })

# Create DataFrame
mule_hops = pd.DataFrame(mule_hops)

# Save
mule_hops.to_csv("mule_hops.csv", index=False)

print("mule_hops.csv created successfully!")

print("\nShape:")
print(mule_hops.shape)

print("\nColumns:")
print(mule_hops.columns.tolist())

print("\nFirst 10 records:")
print(mule_hops.head(10))

mule_hops.csv created successfully!

Shape:
(1500, 8)

Columns:
['hop_id', 'complaint_id', 'hop_level', 'from_account', 'to_account', 'amount_transferred_inr', 'time_delay_minutes', 'timestamp']

First 10 records:
    hop_id complaint_id  hop_level from_account to_account  \
0  HOP0001    CMP100001          1     ACC0001V  ACC0001M1   
1  HOP0002    CMP100001          2    ACC0001M1  ACC0001M2   
2  HOP0003    CMP100001          3    ACC0001M2  ACC0001M3   
3  HOP0004    CMP100002          1     ACC0002V  ACC0002M1   
4  HOP0005    CMP100002          2    ACC0002M1  ACC0002M2   
5  HOP0006    CMP100002          3    ACC0002M2  ACC0002M3   
6  HOP0007    CMP100003          1     ACC0003V  ACC0003M1   
7  HOP0008    CMP100003          2    ACC0003M1  ACC0003M2   
8  HOP0009    CMP100003          3    ACC0003M2  ACC0003M3   
9  HOP0010    CMP100004          1     ACC0004V  ACC0004M1   

   amount_transferred_inr  time_delay_minutes           timestamp  
0                78057.36          

In [ ]:
print(
    mule_hops[
        mule_hops["complaint_id"] == "CMP100001"
    ]
)

    hop_id complaint_id  hop_level from_account to_account  \
0  HOP0001    CMP100001          1     ACC0001V  ACC0001M1   
1  HOP0002    CMP100001          2    ACC0001M1  ACC0001M2   
2  HOP0003    CMP100001          3    ACC0001M2  ACC0001M3   

   amount_transferred_inr  time_delay_minutes           timestamp  
0                78057.36                   5 2026-08-21 10:39:07  
1                73949.08                  12 2026-08-21 10:56:07  
2                69840.79                   8 2026-08-21 11:04:07  


In [ ]:
atm_predictions.columns.tolist()

['id',
 'atm_id',
 'bank_name',
 'city',
 'latitude',
 'longitude',
 'highway_distance',
 'lighting_score',
 'cctv_coverage',
 'historical_fraud_count',
 'withdrawal_limit',
 'risk_score',
 'risk_level',
 'predicted_risk_level',
 'prediction_confidence',
 'date',
 'time',
 'time_of_day',
 'crime_type']

In [ ]:
import pandas as pd

mule_hops = pd.read_csv("mule_hops.csv")
atm_predictions = pd.read_csv("atm_predictions.csv")

print("mule_hops:", mule_hops.shape)
print("atm_predictions:", atm_predictions.shape)

mule_hops: (1500, 8)
atm_predictions: (500, 19)


In [ ]:
complaints = pd.read_csv("complaints.csv")

print(complaints["location"].head(10).tolist())

['Ahmedabad', 'Bangalore', 'Kolkata', 'Pune', 'Delhi', 'Hyderabad', 'Bangalore', 'Delhi', 'Chennai', 'Kolkata']


In [13]:
# Risk priority
risk_priority = {
    "HIGH": 3,
    "MEDIUM": 2,
    "LOW": 1
}

# Calculate priority using existing ML predictions
atm_predictions["risk_priority"] = (
    atm_predictions["predicted_risk_level"].map(risk_priority)
)

atm_predictions["atm_priority_score"] = (
    atm_predictions["risk_priority"] * 50
    + atm_predictions["prediction_confidence"] * 0.3
    + atm_predictions["historical_fraud_count"] * 0.2
)

# Rank ATMs within each city
atm_predictions = atm_predictions.sort_values(
    ["city", "atm_priority_score"],
    ascending=[True, False]
)

atm_predictions["city_rank"] = (
    atm_predictions
    .groupby("city")
    .cumcount() + 1
)

# Keep top 8 existing ATMs per city
top_atms = atm_predictions[
    atm_predictions["city_rank"] <= 8
].copy()

print("Top ATM candidates:", top_atms.shape)

print("\nATMs available per city:")
print(top_atms.groupby("city")["atm_id"].count())

Top ATM candidates: (80, 22)

ATMs available per city:
city
Ahmedabad    8
Bangalore    8
Chennai      8
Delhi        8
Hyderabad    8
Kolkata      8
Mumbai       8
Nagpur       8
Nashik       8
Pune         8
Name: atm_id, dtype: int64


In [14]:
# Get the final cash-out account from Hop 3
cashout_accounts = (
    mule_hops[mule_hops["hop_level"] == 3]
    [["complaint_id", "to_account"]]
    .rename(columns={
        "complaint_id": "complaint_number",
        "to_account": "cashout_account"
    })
)

# Add cash-out account to complaints
complaints_with_cashout = complaints.merge(
    cashout_accounts,
    on="complaint_number",
    how="left"
)

# Create complaint → candidate ATM mapping
mapped_rows = []

for city, city_complaints in complaints_with_cashout.groupby("location"):

    city_atms = (
        top_atms[top_atms["city"] == city]
        .sort_values("atm_priority_score", ascending=False)
        .reset_index(drop=True)
    )

    city_complaints = city_complaints.reset_index(drop=True)

    if len(city_atms) == 0:
        continue

    for i, complaint in city_complaints.iterrows():

        # Rotate through the top 8 existing ATMs
        atm = city_atms.iloc[i % len(city_atms)]

        mapped_rows.append({
            "complaint_number": complaint["complaint_number"],
            "location": complaint["location"],
            "cashout_account": complaint["cashout_account"],
            "atm_id": atm["atm_id"],
            "bank_name": atm["bank_name"],
            "latitude": atm["latitude"],
            "longitude": atm["longitude"],
            "predicted_risk_level": atm["predicted_risk_level"],
            "prediction_confidence": atm["prediction_confidence"]
        })

atm_cashout_mapping = pd.DataFrame(mapped_rows)

print("Mapping created!")
print("Shape:", atm_cashout_mapping.shape)

print("\nUnique ATMs used:")
print(atm_cashout_mapping["atm_id"].nunique())

Mapping created!
Shape: (500, 9)

Unique ATMs used:
80


In [16]:
atm_cashout_mapping.to_csv(
    "atm_cashout_mapping.csv",
    index=False
)

print("atm_cashout_mapping.csv created successfully!")
print("Shape:", atm_cashout_mapping.shape)

atm_cashout_mapping.csv created successfully!
Shape: (500, 9)


In [17]:
with open("atm_relationships.cypher", "w", encoding="utf-8") as f:
    for _, row in atm_cashout_mapping.iterrows():
        f.write(
            f'MATCH (a:Account {{account_id: "{row["cashout_account"]}"}}), '
            f'(atm:ATM {{atm_id: "{row["atm_id"]}"}}) '
            f'MERGE (a)-[r:LIKELY_CASHOUT_AT]->(atm) '
            f'SET r.complaint_number = "{row["complaint_number"]}", '
            f'r.location = "{row["location"]}", '
            f'r.prediction_confidence = {float(row["prediction_confidence"])}, '
            f'r.predicted_risk_level = "{row["predicted_risk_level"]}";\n'
        )

print("atm_relationships.cypher created successfully!")

atm_relationships.cypher created successfully!


In [18]:
from google.colab import files
files.download("atm_relationships.cypher")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>